In [1]:
!pip install -q "transformers>=4.46,<4.50" "torch>=2.4" peft trl bitsandbytes accelerate datasets
import torch, transformers
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
torch: 2.4.1+cu124
transformers: 4.49.0


In [2]:
# =================
# 태스크별 분포 확인
# =================

import os, json
from collections import Counter

rows = [json.loads(l) for l in open('kanana_all_train.jsonl', encoding='utf-8')]
print("train 총:", len(rows))
print("태스크 분포:", dict(Counter(r['meta']['task'] for r in rows)))

# valid도
vrows = [json.loads(l) for l in open('kanana_all_valid.jsonl', encoding='utf-8')]
print("valid 총:", len(vrows))
print("valid 태스크:", dict(Counter(r['meta']['task'] for r in vrows)))

train 총: 451
태스크 분포: {'CON': 150, 'PEP': 151, 'RPT': 150}
valid 총: 44
valid 태스크: {'CON': 15, 'PEP': 14, 'RPT': 15}


In [8]:
for f in ['kanana_all_train.jsonl', 'kanana_all_valid.jsonl']:
    if os.path.exists(f):
        rows = [json.loads(l) for l in open(f, encoding='utf-8') if l.strip()]
        print(f"{f}: {len(rows)}건 ✓")
    else:
        print(f"{f}: ❌ 없음 — 업로드 필요")

kanana_all_train.jsonl: 451건 ✓
kanana_all_valid.jsonl: 44건 ✓


In [9]:
# =================
# 데이터 로드
# =================

from datasets import load_dataset

train_ds = load_dataset("json", data_files="kanana_all_train.jsonl", split="train")
valid_ds = load_dataset("json", data_files="kanana_all_valid.jsonl", split="train")
print(f"train {len(train_ds)} / valid {len(valid_ds)}")
print("샘플 messages 역할:", [m['role'] for m in train_ds[0]['messages']])

train 451 / valid 44
샘플 messages 역할: ['system', 'user', 'assistant']


In [10]:
# ===============================
# 모델 + 토크나이저 (4bit QLoRA)
# ===============================

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

MODEL_ID = "kakaocorp/kanana-1.5-8b-instruct-2505"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 41,943,040 || all params: 8,072,228,864 || trainable%: 0.5196


In [13]:
# ==================================
# 학습 설정 (SFTConfig, 4090 최적화)
# ==================================

from trl import SFTConfig, SFTTrainer

OUT = "kanana_finetuned"

cfg = SFTConfig(
    output_dir=OUT,
    num_train_epochs=3,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,

    # 추가
    per_device_eval_batch_size=1,
    eval_accumulation_steps=1,

    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,

    bf16=True,
    bf16_full_eval=True,

    max_length=3072,

    packing=False,

    save_strategy="epoch", # 에폭마다 저장
    eval_strategy="epoch", # 에폭마다 평가
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    logging_steps=5,
    report_to="none",

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    optim="paged_adamw_8bit",
    max_grad_norm=1.0,
)

In [14]:
# ===================
# 트레이너 + 학습실행
# ===================

trainer = SFTTrainer(
    model=model,
    args=cfg,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    processing_class=tokenizer,
)

trainer.train()

Truncating train dataset:   0%|          | 0/451 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/44 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


Epoch,Training Loss,Validation Loss
1,0.498600,0.484292
2,0.246800,0.454754


/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


TrainOutput(global_step=675, training_loss=0.45058286887628063, metrics={'train_runtime': 1591.0126, 'train_samples_per_second': 0.85, 'train_steps_per_second': 0.424, 'total_flos': 1.0767195580416e+17, 'train_loss': 0.45058286887628063})

In [17]:
# ==========
# 최종 저장
# ==========
trainer.save_model(OUT + "_best")
tokenizer.save_pretrained(OUT + "_best")
print("저장 완료:", OUT + "_best")

# ===========================
# 학습 로그(에폭별 loss) 확인
# ===========================
import pandas as pd
logs = trainer.state.log_history
evals = [l for l in logs if 'eval_loss' in l]
for e in evals:
    print(f"epoch {e['epoch']:.0f}: valid_loss {e['eval_loss']:.4f}")
print(f"\n최종 채택: valid_loss 최저 epoch")

저장 완료: kanana_finetuned_best
epoch 1: valid_loss 0.4843
epoch 2: valid_loss 0.4450
epoch 3: valid_loss 0.4548

최종 채택: valid_loss 최저 epoch
